In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, pipeline
import torch
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

dataset = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")["train"]

def clean_prompt(example):
    example["prompt"] = str(example["prompt"]) if example["prompt"] is not None else ""
    example["A"] = str(example["A"]) if example["A"] is not None else ""
    example["B"] = str(example["B"]) if example["B"] is not None else ""
    example["C"] = str(example["C"]) if example["C"] is not None else ""
    example["D"] = str(example["D"]) if example["D"] is not None else ""
    example["E"] = str(example["E"]) if example["E"] is not None else ""
    return example

dataset = dataset.map(clean_prompt)

def combine(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

dataset = dataset.map(combine)
print("Q1:", len(dataset[51]["combined_text"]))

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Q1: 614


In [3]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print("Q2:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q2: 30522


In [4]:
print("Q3:",tokenizer.sep_token_id)

Q3: 102


In [5]:
prompts_list = [str(p) if p is not None else "" for p in dataset["prompt"]]
encoded = tokenizer(prompts_list, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
print("Q4:", encoded["input_ids"].shape)

Q4: torch.Size([2000, 128])


In [6]:
print("Q5:", 768 // 12)

Q5: 64


In [7]:
model = AutoModel.from_pretrained("bert-base-uncased")
row0 = dataset[0]
inputs0 = tokenizer(row0["prompt"], return_tensors="pt")
with torch.no_grad():
    output0 = model(**inputs0)
print("Q6:", output0.last_hidden_state.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q6: torch.Size([1, 31, 768])


In [8]:
cls_vector = output0.last_hidden_state[0][0]
print("Q7:", round(cls_vector[:5].sum().item(), 4))

Q7: -1.2001


In [9]:
attn_model = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
text = "Light-ion fusion is a technique."
attn_inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    attn_output = attn_model(**attn_inputs)
tokens = tokenizer.convert_ids_to_tokens(attn_inputs["input_ids"][0])
fusion_idx = tokens.index("fusion")
last_layer_attn = attn_output.attentions[-1]
weight = last_layer_attn[0, 0, 0, fusion_idx]
print("Q8:", round(weight.item(), 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q8: 0.1025


In [10]:
st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
row0 = dataset[0]
emb_prompt = st_model.encode(row0["prompt"])
emb_b = st_model.encode(row0["B"])
sim = util.cos_sim(emb_prompt, emb_b)
print("Q9:", round(sim.item(), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Q9: 0.7658


In [11]:
def apk(actual, predicted, k=3):
    if len(predicted) > k:
        predicted = predicted[:k]
    score = 0.0
    hits = 0.0
    for i, p in enumerate(predicted):
        if p == actual:
            hits += 1.0
            score += hits / (i + 1.0)
            break
    return score

def mapk(actuals, predicteds, k=3):
    return np.mean([apk(a, p, k) for a, p in zip(actuals, predicteds)])

options = ["A", "B", "C", "D", "E"]
answers = dataset["answer"]

tfidf_top3_all = []
for i in range(len(dataset)):
    row = dataset[i]
    corpus = [row["prompt"]] + [row[o] for o in options]
    vect = TfidfVectorizer().fit_transform(corpus)
    sims = cosine_similarity(vect[0:1], vect[1:]).flatten()
    ranked = [options[j] for j in np.argsort(-sims)]
    tfidf_top3_all.append(ranked[:3])

minilm_top3_all = []
for i in range(len(dataset)):
    row = dataset[i]
    prompt_emb = st_model.encode(row["prompt"])
    option_embs = st_model.encode([row[o] for o in options])
    sims = util.cos_sim(prompt_emb, option_embs).numpy().flatten()
    ranked = [options[j] for j in np.argsort(-sims)]
    minilm_top3_all.append(ranked[:3])

map3_minilm = mapk(answers, minilm_top3_all, k=3)
print("Q10a:", map3_minilm)

Q10a: 0.4230833333333333


In [12]:
count = 0
for a, tf3, mn3 in zip(answers, tfidf_top3_all, minilm_top3_all):
    if a not in tf3 and a in mn3:
        count += 1
print("Q10b:", count)

Q10b: 522


In [13]:
zsc = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
row1 = dataset[1]
candidate_labels = [row1["A"], row1["B"], row1["C"]]
result = zsc(row1["prompt"], candidate_labels)
top_score = result["scores"][0]
print("Q11:", round(top_score, 4))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q11: 0.4575


In [14]:
result_multi = zsc(row1["prompt"], candidate_labels, multi_label=True)
diff = abs(sum(result["scores"]) - sum(result_multi["scores"]))
print("Q12:", round(diff, 4))

Q12: 0.9995


In [15]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

t5_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
t5_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

row0 = dataset[0]
prompt_str = f"Question: {row0['prompt']}. Is the correct answer A: {row0['A']} or B: {row0['B']}? Answer with just the letter A or B."

t5_inputs = t5_tokenizer(prompt_str, return_tensors="pt")
t5_output_ids = t5_model.generate(**t5_inputs, max_new_tokens=5)
t5_output_text = t5_tokenizer.decode(t5_output_ids[0], skip_special_tokens=True)
print("Q13:", t5_output_text)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Q13: B
